In [6]:
# Part 1 - Create the Project List with CI Platform Detection

import os
import re
import yaml
import pandas as pd

# === CONFIG ===
PROJECTS_DIR = r"C:\Users\Admin\OneDrive\Education\Master of Info - Thesis\Mobile App Data\Config Files"
OUTPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "Project_List.csv")

# === CLASSIFICATION KEYWORDS ===
TEST_TYPES = {
    'firebase_Full': ['gcloud firebase test android run'],
    'firebase_Compact': ['Firebase-Test-Lab-Action'],
    'appcenter_test': ['appcenter test run', 'microsoft/appcenter-test-cli-action'],
    'browserstack_test': ['browserstack', 'browserstack/github-actions'],
    'GitHub_emulator_full': ['android-emulator-runner'],
    'GitHub_emulator_compact': ['malinskiy/action-android/emulator-run-cmd'],
    'emulator_manual': ['create avd'],
    'GitHub_GMD': ['cleanManagedDevices'],
    'Unit_Test': ['gradlew test', './gradlew test', 'testDebugUnitTest', 'testReleaseUnitTest',
                  'test', 'run unit tests', 'run: test', 'npm test', 'yarn test'],
    'Other': ['instrumentation', 'instrument']
}

# === CI PLATFORM DETECTION PATTERNS (IMPROVED) ===
CI_PLATFORM_PATTERNS = {
    "GitHub Actions": [r"uses:\s*\S+/", r"runs-on:\s*\S+"],
    "GitLab CI": [r"only:", r"tags:", r"gitlab-ci"],
    "Bitrise": [r"bitrise-steplib", r"bitriseio"],
    "CircleCI": [r"orbs:", r"circleci", r"circleci.com", r"executor:"],
    "Travis CI": [r"language:\s*android", r"dist:\s*\S+"],
    "Azure Pipelines": [r"vmImage:", r"pool:"],
    "Jenkins": [r"pipeline\s*{", r"agent\s+any"],
    "CodeMagic": [r"flutter:", r"codemagic.yaml", r"workflow:"]
}

# === STRUCTURES TO HOLD RESULTS ===
project_results = {}
ci_platforms_dict = {}

# === DETECTION LOGIC ===
def detect_testing_types(yaml_text):
    uncommented_text = '\n'.join(
        line for line in yaml_text.splitlines()
        if not line.strip().startswith('#')
    ).lower()

    found = set()
    for label, keywords in TEST_TYPES.items():
        for kw in keywords:
            if kw.lower() in uncommented_text:
                found.add(label)
    return found

# === MAIN PARSER ===
def parse_yaml_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw = f.read().replace('\t', ' ')
            detected = detect_testing_types(raw)
            content = yaml.safe_load(raw)
            if not content:
                return {'types': detected, 'error': True, 'content': raw}
            return {'types': detected, 'error': False, 'content': raw}
    except Exception:
        return {'types': set(), 'error': True, 'content': ''}

# === PROJECT SCANNER ===
for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)
            parts = filename.split(".")
            project_name = (parts[1] if len(parts) > 2 else parts[0]).lower()

            result = parse_yaml_file(file_path)

            if project_name not in project_results:
                project_results[project_name] = {'types': set(), 'errors': 0}
            project_results[project_name]['types'].update(result['types'])
            if result['error']:
                project_results[project_name]['errors'] += 1

            # === CI PLATFORM DETECTION ===
            file_path_lower = file_path.replace("\\", "/").lower()
            found_platforms = set()

            # Filename-based clues
            if ".github/workflows/" in file_path_lower:
                found_platforms.add("GitHub Actions")
            if ".gitlab-ci.yml" in file_path_lower:
                found_platforms.add("GitLab CI")
            if "bitrise.yml" in file_path_lower:
                found_platforms.add("Bitrise")
            if ".circleci/config.yml" in file_path_lower:
                found_platforms.add("CircleCI")
            if ".travis.yml" in file_path_lower:
                found_platforms.add("Travis CI")
            if "azure-pipelines.yml" in file_path_lower:
                found_platforms.add("Azure Pipelines")
            if "jenkinsfile" in file_path_lower:
                found_platforms.add("Jenkins")
            if "codemagic.yaml" in file_path_lower:
                found_platforms.add("CodeMagic")

            # Content-based clues
            for platform, patterns in CI_PLATFORM_PATTERNS.items():
                for pattern in patterns:
                    if re.search(pattern, result['content'], re.IGNORECASE):
                        found_platforms.add(platform)
                        break

            if project_name not in ci_platforms_dict:
                ci_platforms_dict[project_name] = set()
            ci_platforms_dict[project_name].update(found_platforms)


# === Filtered CI Platforms Logic ===
ci_platforms_modified_dict = {}

for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            file_path_lower = file_path.replace("\\", "/").lower()
            filename = os.path.basename(file_path)
            parts = filename.split(".")
            project_name = (parts[1] if len(parts) > 2 else parts[0]).lower()

            result = parse_yaml_file(file_path)
            test_types = result['types']
            content = result['content']

            # Skip if only Unit_Test or none
            effective_types = test_types - {'Unit_Test', 'Other'}
            if not effective_types:
                continue

            found_platforms = set()

            # Filename-based clues
            if ".github/workflows/" in file_path_lower:
                found_platforms.add("GitHub Actions")
            if ".gitlab-ci.yml" in file_path_lower:
                found_platforms.add("GitLab CI")
            if "bitrise.yml" in file_path_lower:
                found_platforms.add("Bitrise")
            if ".circleci/config.yml" in file_path_lower:
                found_platforms.add("CircleCI")
            if ".travis.yml" in file_path_lower:
                found_platforms.add("Travis CI")
            if "azure-pipelines.yml" in file_path_lower:
                found_platforms.add("Azure Pipelines")
            if "jenkinsfile" in file_path_lower:
                found_platforms.add("Jenkins")
            if "codemagic.yaml" in file_path_lower:
                found_platforms.add("CodeMagic")

            # Content-based clues
            for platform, patterns in CI_PLATFORM_PATTERNS.items():
                for pattern in patterns:
                    if re.search(pattern, content, re.IGNORECASE):
                        found_platforms.add(platform)
                        break

            if project_name not in ci_platforms_modified_dict:
                ci_platforms_modified_dict[project_name] = set()
            ci_platforms_modified_dict[project_name].update(found_platforms)




# === EXPORT CSV ===
rows = []
for project, result in project_results.items():
    cleaned_types = result['types']
    if 'Other' in cleaned_types and len(cleaned_types) > 1:
        cleaned_types = cleaned_types - {'Other'}

    rows.append({
        'project': project,
        'test_types': ', '.join(sorted(cleaned_types)) if cleaned_types else 'none',
        'yaml_errors': result['errors'],
        'ci_platforms': ', '.join(sorted(ci_platforms_dict.get(project, []))) if project in ci_platforms_dict else 'none',
        'ci_platforms_Modified': ', '.join(sorted(ci_platforms_modified_dict.get(project, []))) if project in ci_platforms_modified_dict else 'none'
    })
    
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
Project_List = df.copy()

print(f"\n✅ Summary written to: {OUTPUT_CSV}")



✅ Summary written to: C:\GitHub\Android-Mobile-Apps\Project_List.csv


In [4]:

# Part 2 - adding the requested columns to Project List
import os
import pandas as pd

PROJECTS_DIR = r"C:\Users\Admin\OneDrive\Education\Master of Info - Thesis\Config Files"  # desktop
OUTPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "Project_List.csv")

#read from table above
df = Project_List.copy()

# 0 - Unit Test: True if any test_type include "Unit_Test"
df['Unit Test'] = df['test_types'].apply(
    lambda x: any(item.strip().startswith('Unit_Test') for item in x.split(','))
)


# 1 - Instrumentation Testing: True if test_type is not "none" nor only unit_test is detected
df['Instrumentation Testing'] = df['test_types'].apply(
    lambda x: x.strip().lower() != 'none' and not (
        len([t for t in x.split(',') if t.strip()]) == 1 and x.strip() == 'Unit_Test'
    )
)

# 2 - GitHub Action: True if any test_type starts with "GitHub"
df['GitHub Action'] = df['test_types'].apply(
    lambda x: any(item.strip().startswith('GitHub') for item in x.split(','))
)

# 3 - GitHub Action Type: include all GitHub-related test types
df['GitHub Action Type'] = df['test_types'].apply(
    lambda x: ', '.join([item.strip() for item in x.split(',') if item.strip().startswith('GitHub')])
)

# 4 - Third_Party: True if any test_type does not start with "GitHub" and is not "none" or "other" or "unit_test"
df['Third_Party'] = df['test_types'].apply(
    lambda x: any(
        not item.strip().startswith('GitHub') and item.strip().lower() not in ['none', 'other','unit_test']
        for item in x.split(',')
    )
)

# 5 - Third_Party_Name: list the test types that do not start with "GitHub" and are not "none" or "other" or "unit_test"
df['Third_Party_Name'] = df['test_types'].apply(
    lambda x: ', '.join([
        item.strip() for item in x.split(',')
        if not item.strip().startswith('GitHub') and item.strip().lower() not in ['none', 'other','unit_test']
    ])
)


# Save it back to CSV if needed

df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Summary written to: {OUTPUT_CSV}")


# Optional: assign to Total_Projects for in-memory use
Total_Projects = df



✅ Summary written to: C:\GitHub\Android-Mobile-Apps\Project_List.csv
